# Palmyra X4 and X5 — a family where every model answers differently

Writer has three models on Amazon Bedrock, and they disagree on almost every
operational question. This is the clearest example in the collection of why
"which provider" is the wrong unit of decision.

| | `palmyra-vision-7b` | `palmyra-x4` / `palmyra-x5` |
|---|---|---|
| Endpoints | `bedrock-mantle` **and** `bedrock-runtime` | `bedrock-runtime` **only** |
| Model ID | bare ID works | **inference profile required** (`us.` prefix) |
| System prompt on Converse | **rejected** | accepted |
| Tool use | **not supported** | supported |

So within one provider: one model you can reach two ways and one you cannot,
one that needs a profile and one that does not, one that refuses a system prompt
and one that takes it, one with tools and one without.

The sibling notebook covers `palmyra-vision-7b`. This one covers the text models,
and every claim in the table above is verified below.


In [1]:
import sys

sys.path.insert(0, "../_shared")

from bedrock import (
    converse,
    converse_tool_uses,
    endpoints_for,
    list_models,
    resolve_runtime_id,
)

REGION = "us-east-1"
X5 = "writer.palmyra-x5-v1"
X4 = "writer.palmyra-x4-v1"
VISION = "writer.palmyra-vision-7b"

for model in (VISION, X4, X5):
    print(f"{model:<28} {endpoints_for(model)}")

print()
print("Writer models on bedrock-mantle:",
      sorted(m for m in list_models(REGION) if m.startswith("writer.")))
print("=> the text models are simply not there. Converse is the only way in.")


writer.palmyra-vision-7b     {'mantle': True, 'runtime': True}


writer.palmyra-x4-v1         {'mantle': False, 'runtime': True}


writer.palmyra-x5-v1         {'mantle': False, 'runtime': True}



Writer models on bedrock-mantle: ['writer.palmyra-vision-7b']
=> the text models are simply not there. Converse is the only way in.


## 1. The text models need an inference profile

`palmyra-x4` and `palmyra-x5` are `INFERENCE_PROFILE`-only. Called by their bare
model ID they are refused — the error names on-demand throughput explicitly,
which is the tell for this whole class of model.


In [2]:
from bedrock import runtime_client

for model in (X4, X5):
    print(f"{model}")
    print(f"    resolves to: {resolve_runtime_id(model, REGION)}")

# What the bare ID actually returns, so you recognise the error.
try:
    runtime_client(REGION).converse(
        modelId=f"{X5}:0",  # bare, no us. prefix
        messages=[{"role": "user", "content": [{"text": "Reply OK"}]}],
        inferenceConfig={"maxTokens": 16},
    )
    print("\nbare ID: accepted")
except Exception as exc:
    print(f"\nbare ID: {type(exc).__name__}")
    print(f"    {str(exc)[-150:]}")


writer.palmyra-x4-v1


    resolves to: us.writer.palmyra-x4-v1:0
writer.palmyra-x5-v1
    resolves to: us.writer.palmyra-x5-v1:0



bare ID: ValidationException
    ter.palmyra-x5-v1:0 with on-demand throughput isn’t supported. Retry your request with the ID or ARN of an inference profile that contains this model.


## 2. A working call, and the system prompt that Palmyra Vision refuses

The same request shape that `palmyra-vision-7b` rejects works here. Worth trying
both ways round on any new model rather than assuming the family behaves alike.


In [3]:
for label, kwargs in [
    ("no system prompt", {}),
    ("with system prompt", {"system": "You are terse. Answer in one sentence."}),
]:
    text, response = converse(
        X5,
        [{"role": "user", "content": [{"text": "Why is idempotency useful?"}]}],
        max_tokens=300,
        region=REGION,
        **kwargs,
    )
    error = (response.get("error") or {}).get("message")
    if error:
        print(f"{label:<20} FAILED {error[:90]}")
    else:
        print(f"{label:<20} {response.get('stopReason'):<10} "
              f"{len(text.split()):>3} words  {text.strip()[:80]}")

print()
print("For contrast, the same system prompt against palmyra-vision-7b:")
text, response = converse(
    VISION,
    [{"role": "user", "content": [{"text": "Why is idempotency useful?"}]}],
    system="You are terse.",
    max_tokens=60,
    region=REGION,
)
error = (response.get("error") or {}).get("message")
print("   ", error[:130] if error else f"accepted: {text.strip()[:80]}")


no system prompt     end_turn   144 words  Idempotency is super useful because it ensures that no matter how many times you


with system prompt   end_turn    24 words  Idempotency ensures that repeating an operation has the same effect as doing it 

For contrast, the same system prompt against palmyra-vision-7b:


    The model returned the following errors: {"error":{"code":"validation_error","message":"ErrorEvent { error: APIError { type: \"Bad


## 3. Tool use works on the text models

`palmyra-vision-7b` has no tool support at all, which the sibling notebook works
around. The text models do support tools, in the Converse `toolSpec` shape —
note it is not the OpenAI shape, and the JSON schema nests under `inputSchema.json`.


In [4]:
TOOLS = [
    {
        "toolSpec": {
            "name": "multiply",
            "description": "Multiply two integers",
            "inputSchema": {
                "json": {
                    "type": "object",
                    "properties": {
                        "a": {"type": "integer"},
                        "b": {"type": "integer"},
                    },
                    "required": ["a", "b"],
                }
            },
        }
    }
]

text, response = converse(
    X5,
    [{"role": "user", "content": [{"text": "What is 17 * 23? Use the tool."}]}],
    max_tokens=300,
    tools=TOOLS,
    region=REGION,
)
error = (response.get("error") or {}).get("message")
if error:
    print("failed:", error[:150])
else:
    print("stop reason:", response.get("stopReason"))
    for use in converse_tool_uses(response):
        print("tool call  :", use["name"], use["input"])
        # Verify the model extracted the right operands rather than trusting it.
        got = use["input"]
        correct = {got.get("a"), got.get("b")} == {17, 23}
        print("operands   :", "correct" if correct else f"WRONG {got}")


stop reason: tool_use
tool call  : multiply {'a': 17, 'b': 23}
operands   : correct


## 4. X4 or X5?

X5 supersedes X4. Both are still listed, so the useful question is whether the
older one is ever the right choice — compare them on the same prompt and let the
token counts and answers decide rather than the version number.


In [5]:
PROMPT = "In one sentence, when should a queue be preferred over a direct call?"

print(f"{'model':<26} {'stop':<12} {'tokens':>7}  answer")
print("-" * 96)
for model in (X4, X5):
    text, response = converse(
        model,
        [{"role": "user", "content": [{"text": PROMPT}]}],
        system="You are terse.",
        max_tokens=150,
        region=REGION,
    )
    error = (response.get("error") or {}).get("message")
    if error:
        print(f"{model:<26} ERROR {error[:50]}")
        continue
    total = response.get("usage", {}).get("totalTokens", 0)
    print(f"{model:<26} {response.get('stopReason'):<12} {total:>7}  {text.strip()[:52]}")


model                      stop          tokens  answer
------------------------------------------------------------------------------------------------


writer.palmyra-x4-v1       end_turn          46  Use a queue when you need to manage tasks asynchrono


writer.palmyra-x5-v1       end_turn          55  Use a queue over a direct call when you need to deco


## 5. What to take away

- **The endpoint answer can differ inside one provider.** Palmyra Vision is on both
  endpoints; the text models are on `bedrock-runtime` only. Deciding "we will use
  `bedrock-mantle` for Writer" would silently exclude two of their three models.
- **`INFERENCE_PROFILE`-only is a real category.** The bare model ID fails with an
  error about on-demand throughput. `resolve_runtime_id()` handles it, but you
  should recognise the message, because most of the Claude family behaves the
  same way.
- **Do not generalise a limitation across a family.** Palmyra Vision refuses a
  system prompt and has no tools. Both facts are false for X4 and X5.
- **Check tool arguments, not just that a tool was called.** Section 3 asserts the
  operands are 17 and 23. A tool call with the wrong arguments still returns
  `stopReason: tool_use`, so counting tool calls proves nothing on its own.
